# Historical Data with Pandas
Resolve a visible historical price and calculate a teaching risk proxy.

## 1. Load synthetic prices

In [ ]:
import pandas as pd

prices = pd.read_csv("data/sample_prices.csv")
prices.head()

## 2. Inspect the table

In [ ]:
print(prices.columns.tolist())
print(prices.dtypes)

## 3. Convert dates

In [ ]:
prices["date"] = pd.to_datetime(prices["date"])
print(prices["date"].dtype)

## 4. Filter for SPY

In [ ]:
spy = prices[prices["symbol"] == "SPY"]
print(spy.head())

## 5. Limit data to the simulated date

In [ ]:
simulated_date = pd.Timestamp("2020-03-20")
visible_spy = spy[spy["date"] <= simulated_date]
print(visible_spy.tail())

## 6. Resolve the last visible price

In [ ]:
latest_row = visible_spy.sort_values("date").iloc[-1]
print(latest_row[["date", "close"]])

## 7. A weekend uses Friday

In [ ]:
weekend = pd.Timestamp("2020-03-22")
weekend_row = spy[spy["date"] <= weekend].sort_values("date").iloc[-1]
print(weekend_row[["date", "close"]])

## 8. Pivot symbols into columns

In [ ]:
wide = prices.pivot(index="date", columns="symbol", values="close").sort_index()
print(wide.head())

## 9. Calculate daily returns

In [ ]:
visible_wide = wide[wide.index <= simulated_date]
returns = visible_wide.pct_change().dropna()
print(returns.head())

## 10. Calculate covariance

In [ ]:
covariance = returns.cov()
print(covariance)

## 11. Apply fixed weights

In [ ]:
import numpy as np

weights = np.array([0.20, 0.30, 0.50])  # GLD, QQQ, SPY column order
annualized_volatility = np.sqrt(weights @ covariance.values @ weights) * np.sqrt(252)
print("Historical teaching proxy:", annualized_volatility)

## 12. Deliberate limitation: future leakage

In [ ]:
leaky_last_date = spy.sort_values("date").iloc[-1]["date"]
print("Wrong because it sees the future:", leaky_last_date)

## 13. Correct the leakage

In [ ]:
correct_last_date = spy[spy["date"] <= simulated_date].sort_values("date").iloc[-1]["date"]
print("Visible date only:", correct_last_date)

## Takeaways
- Date filters define what the simulation can know.
- Returns and covariance support a simple risk calculation.
- Volatility here is a historical teaching proxy, not a forecast.